# ShopMind-AI — 03 — Stage 3, the purchasing plan

**Course:** ML Semester Project, Fall 2026
**Seed:** 42 everywhere (`.seed(42)`, `random_state=42`, `GeneratorConfig(seed=42)`).
**Data:** self-generated Pakistani seller panel (`data/demo/`) plus the real *Pakistan's
Largest E-Commerce Dataset* (`data/processed/`). Sources and licences are in
`data/demo/DATA_CARD.md` and `docs/PAKISTAN_DATASET.md`.

This notebook is **executed top-to-bottom** and saved with its outputs, so every number
shown below is reproducible by running `python scripts/build_notebooks.py`.

| Notebook | Question it answers |
|---|---|
| `01_eda.ipynb` | What does the data look like, and what would a naive guess score? |
| `02_modelling.ipynb` | Which model wins, on which window, and by how much? |
| `03_planner.ipynb` | Given a budget and a buy list, what should the seller purchase? |

## 1. The decision problem

The seller supplies a CSV with the products they intend to buy, each with a **current
purchase price** and the **units already available**, plus a **budget**. The system must
return a purchase plan.

The naive answer — buy the products with the highest demand — is wrong, because demand is
not profit. This notebook shows the ranking, the allocation, and what happens when the
assumptions are violated.

## 2. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
np.random.seed(42)
pd.set_option('display.width', 150)

from src.planning import expected_profit_per_unit, forecast_request, summarise
from src.modeling import BaseModel

model = BaseModel.load(str(ROOT / 'models/base_model.pkl'))
print('loaded model:', model.name)

loaded model: Gradient Boosting


## 3. Unit economics — where profit actually comes from

A unit is profitable only if

```
retail price
  - purchase cost
  - marketplace commission        (per category)
  - shipping + COD fee
  - expected return cost          (return rate x full cost of a bounced parcel)
  - allocated ad spend
> 0
```

Returns are the term that surprises people: at a 21% return rate on fashion, a 2x markup
can still be loss-making.

### 3.1 The trap: a 5,000-cost item at a 5,100 retail price

In [2]:
rows = []
for retail in [5100, 6000, 7000, 8000, 9500]:
    rows.append({'retail': retail, 'cost': 5000,
                 'profit_per_unit': round(expected_profit_per_unit(retail, 5000, 'Home & Kitchen'), 2)})
pd.DataFrame(rows)

,retail,cost,profit_per_unit
0,5100,5000,-873.2
1,6000,5000,-200.0
2,7000,5000,548.0
3,8000,5000,1296.0
4,9500,5000,2418.0


**Takeaway.** At a 5,100 retail on a 5,000 landed cost the item loses **Rs 873 per unit**.
The seller is "making a sale" and destroying money. This is why the planner ranks by
`expected_profit_per_unit / CurrentPrice` and not by units or by revenue.

## 4. Load the seller's buy list

In [3]:
request = pd.read_csv(ROOT / 'data/demo/seller_purchase_request.csv')
history = pd.read_csv(ROOT / 'data/demo/seller_sales_history.csv', parse_dates=['month'])

print(f'{len(request)} products to consider, budget will be applied below')
request.head(8)

36 products to consider, budget will be applied below


,ProductID,ProductName,CurrentPrice,RetailPrice,CurrentStock,Category,UnitsWanted
0,P028,Air Fryer - 5L,12500.58,21679.60,0,Home & Kitchen,5
1,P003,Yoga Mat - 8mm,1695.46,2742.53,5,Sports & Fitness,20
2,P042,Baby Fleece Blanket - Large,1026.51,2130.95,13,Baby & Kids,123
3,P059,Sunblock SPF 50 - 75ml,767.17,1502.75,42,Beauty & Personal Care,107
4,P019,Mini Bluetooth Speaker - 5W,2449.38,3444.63,4,Electronics,25
5,P052,Skipping Rope - Basic,243.10,504.90,7,Sports & Fitness,35
6,P054,Skipping Rope - Speed,526.05,940.95,24,Sports & Fitness,33
7,P018,Kids Water Bottle - 800ml,813.70,1480.45,25,Baby & Kids,76


## 5. Produce a plan for a Rs 500,000 budget

In [4]:
BUDGET = 500_000.0

result = forecast_request(request, model, BUDGET, history=history,
                          forecast_month='2026-10')
plan = result.plan
stats = summarise(plan, BUDGET)

for k, v in stats.items():
    print(f'{k:<28} {v}')
print()
print('assumptions the system is making:')
for a in result.assumptions:
    print(' -', a)

budget                       500000.0
investment                   499884.09
unallocated                  115.91
utilisation_pct              100.0
expected_profit              252746.59
expected_roi_pct             50.6
products                     36
funded_products              11
partial_fill_products        24
unprofitable_products        4
skipped_below_margin         0

assumptions the system is making:
 - Demand is forecast as a ratio to the trailing three-month mean and converted back to units.
 - Safety stock = 1.28 x the sample standard deviation of the last six months (about a 90% service level under a normal demand assumption).
 - Profit per unit = price x (1 - return rate) - marketplace commission - ad spend - outbound shipping - return handling - unit cost.
 - Budget is allocated greedily by expected profit per rupee of purchase; it is not a globally optimal knapsack solution.


In [5]:
cols = ['ProductID', 'ProductName', 'Category', 'CurrentPrice', 'CurrentStock',
        'AvgDemand', 'SafetyStock', 'RequiredInventory', 'ShortageUnits',
        'ProfitPerUnit', 'ProfitPerRupee', 'RecommendedPurchase', 'Investment',
        'ExpectedProfit', 'IsUnprofitable']
shown = [c for c in cols if c in plan.columns]
plan[shown].sort_values('Investment', ascending=False).head(12).round(2)

,ProductID,ProductName,Category,CurrentPrice,CurrentStock,AvgDemand,SafetyStock,RequiredInventory,ShortageUnits,ProfitPerUnit,ProfitPerRupee,RecommendedPurchase,Investment,ExpectedProfit,IsUnprofitable
0,P033,3-Piece Chiffon Suit - Bridal,Fashion,7630.59,0,33.31,11.78,45.09,46,3425.47,0.45,17,129720.03,58233.00,False
1,P032,3-Piece Chiffon Suit - Formal,Fashion,5747.45,14,24.06,10.25,34.32,21,3019.93,0.53,21,120696.45,63418.49,False
2,P042,Baby Fleece Blanket - Large,Baby & Kids,1026.51,13,69.62,19.20,88.82,76,582.61,0.57,76,78014.76,44278.01,False
3,P056,Baby Diaper Pants (Pack) - Medium,Baby & Kids,1048.02,14,40.35,28.35,68.69,55,503.02,0.48,55,57641.10,27666.13,False
4,P057,Baby Diaper Pants (Pack) - Large,Baby & Kids,1170.22,16,29.85,19.48,49.33,34,592.85,0.51,34,39787.48,20157.01,False
5,P036,2-in-1 Juicer Blender - 4-in-1,Home & Kitchen,7541.84,4,5.60,2.77,8.36,5,4073.44,0.54,5,37709.20,20367.18,False
6,P041,Baby Fleece Blanket - Medium,Baby & Kids,682.73,90,66.56,44.35,110.91,21,379.14,0.56,21,14337.33,7962.03,False
7,P037,Dumbbell Set - 5kg,Sports & Fitness,2483.03,5,4.85,3.47,8.33,4,1224.62,0.49,4,9932.12,4898.50,False
8,P046,Vitamin C Serum - 15ml,Beauty & Personal Care,600.52,133,122.60,25.89,148.48,16,307.81,0.51,16,9608.32,4925.00,False
9,P066,Herbal Face Wash - Combo,Beauty & Personal Care,524.13,1,74.28,68.43,142.71,142,184.54,0.35,4,2096.52,738.16,False


## 6. Budget sensitivity — the plan is a curve, not a single answer

A useful decision system tells the seller what the *next* rupee buys.

In [6]:
curve = []
for budget in [100_000, 250_000, 500_000, 750_000, 1_000_000, 1_500_000]:
    r = forecast_request(request, model, budget, history=history, forecast_month='2026-10')
    s = summarise(r.plan, budget)
    curve.append({'budget': budget,
                  'investment': s['investment'],
                  'unallocated': s['unallocated'],
                  'expected_profit': s['expected_profit'],
                  'expected_roi_pct': s['expected_roi_pct'],
                  'funded_products': s['funded_products'],
                  'unprofitable_products': s['unprofitable_products']})
pd.DataFrame(curve).round(1)

,budget,investment,unallocated,expected_profit,expected_roi_pct,funded_products,unprofitable_products
0,100000,99893.9,106.1,56313.5,56.4,3,4
1,250000,249814.4,185.6,135468.3,54.2,5,4
2,500000,499884.1,115.9,252746.6,50.6,11,4
3,750000,749964.8,35.2,363312.8,48.4,13,4
4,1000000,999938.8,61.2,459665.3,46.0,16,4
5,1500000,1499955.7,44.3,611879.9,40.8,27,4


**Takeaway.** Return on budget falls as the budget grows, because the best
profit-per-rupee items are funded first and the marginal product is worse. That shape is
the practical answer to "how much stock should I buy?" — the seller should stop adding
budget where the marginal ROI drops below their cost of capital.

## 7. What would make the plan wrong

The rubric rewards an explicit statement of limitations. The plan is conditional on:

1. **The demand forecast**, which notebook 02 shows does not beat a trailing mean.
2. **The return rate assumption**, which the real Pakistani data shows could be 2–4× higher.
3. **Prices holding** for the purchase lead time.
4. **Greedy allocation** — it is not an integer program, so it is near-optimal, not optimal.

A stress test on (2) below, since it is the largest risk.

In [7]:
# `forecast_request` accepts an explicit return-rate override, so the stress test does
# not have to mutate module state. 10% is the planning default for fashion-adjacent
# categories; 48% is what the real Pakistani dataset actually shows.
rows = []
for rate in [0.10, 0.21, 0.30, 0.48, 0.60]:
    r = forecast_request(request, model, BUDGET, history=history,
                         forecast_month='2026-10', return_rate=rate)
    s = summarise(r.plan, BUDGET)
    rows.append({'assumed_return_rate': f'{rate:.0%}',
                 'median_profit_per_unit': round(float(r.plan.ProfitPerUnit.median()), 0),
                 'investment': round(s['investment'], 0),
                 'expected_profit': round(s['expected_profit'], 0),
                 'expected_roi_pct': s['expected_roi_pct'],
                 'funded_products': s['funded_products'],
                 'unprofitable_products': s['unprofitable_products']})
pd.DataFrame(rows)

,assumed_return_rate,median_profit_per_unit,investment,expected_profit,expected_roi_pct,funded_products,unprofitable_products
0,10%,449.0,499922.0,276415.0,55.3,8,3
1,21%,363.0,499984.0,236737.0,47.3,7,4
2,30%,294.0,499991.0,205312.0,41.1,7,4
3,48%,154.0,499967.0,146360.0,29.3,5,12
4,60%,61.0,499923.0,107165.0,21.4,5,16


**Takeaway.** The plan's profitability is more sensitive to the return assumption than to
anything the model does. At the planning default the same Rs 500,000 returns a healthy
profit; at the return rate the real marketplace actually exhibits, the funded set shrinks
and the unprofitable count grows. A seller who plans on the optimistic number will buy
stock that cannot be sold profitably.

## 8. Conclusion

The planner is the part of the system that survives the negative forecasting result. Even
with a mediocre demand forecast, ranking by profit-per-rupee and respecting a hard budget
produces a defensible purchase decision — and it correctly refuses to fund items whose
retail price does not clear their fully-loaded cost.